# SVM: Feature-Set Expansion

This notebook evaluates how the SVM classifier responds to the four Gaia XP feature blocks used in this experiment: normalized coefficients (`c`), derivative coefficients (`d`), coefficient uncertainties (`err`), and coefficient-level signal-to-noise features (`snr`). It loads the packaged raw run table by default, builds summary tables, and produces a compact set of standardized figures.

## Configuration and Styling

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
MODEL_NAME = "SVM"
MODEL_SLUG = "svm"

RAW_RESULTS_PATH = BASE_DIR / "precomputed_results" / "feature_set_expansion_raw_results.parquet"
OUT_DIR = BASE_DIR / f"{MODEL_SLUG}_feature_set_out"
FIGURE_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
for path in [OUT_DIR, FIGURE_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

USE_PRECOMPUTED_RESULTS = True
RUN_FULL_SWEEP = False
BASELINE_COMBO = "c"
BLOCK_ORDER = ["c", "d", "err", "snr"]

METRICS = {
    "test_PR_AUC": {"label": "PR-AUC", "higher_is_better": True},
    "test_ROC_AUC": {"label": "ROC-AUC", "higher_is_better": True},
    "test_LogLoss": {"label": "Log loss", "higher_is_better": False},
    "test_Brier": {"label": "Brier score", "higher_is_better": False},
}
THRESHOLD_METRICS = {
    "youden_test_Precision": {"label": "Youden precision", "higher_is_better": True},
    "youden_test_Sensitivity": {"label": "Youden sensitivity", "higher_is_better": True},
    "youden_test_Specificity": {"label": "Youden specificity", "higher_is_better": True},
    "f1_test_Precision": {"label": "F1 precision", "higher_is_better": True},
    "f1_test_Sensitivity": {"label": "F1 sensitivity", "higher_is_better": True},
    "f1_test_Specificity": {"label": "F1 specificity", "higher_is_better": True},
}

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.titlesize": 12,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#D8DEE9",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.65,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
})

COLORS = {
    "baseline": "#1F2937",
    "one": "#6B7280",
    "two": "#2B6CB0",
    "three": "#2F855A",
    "four": "#7F56D9",
    "ci": "#CBD5E1",
    "accent": "#0F766E",
    "negative": "#B91C1C",
}

print("MODEL:", MODEL_NAME)
print("RAW_RESULTS_PATH:", RAW_RESULTS_PATH)
print("OUT_DIR:", OUT_DIR)

## Shared Analysis Helpers

In [ ]:
def parse_combo(combo: str) -> list[str]:
    if pd.isna(combo) or combo == "":
        return []
    return [part for part in str(combo).split("+") if part]


def combo_label(combo: str) -> str:
    return " + ".join(parse_combo(combo))


def combo_color(combo: str) -> str:
    if combo == BASELINE_COMBO:
        return COLORS["baseline"]
    n_blocks = len(parse_combo(combo))
    return {1: COLORS["one"], 2: COLORS["two"], 3: COLORS["three"], 4: COLORS["four"]}.get(n_blocks, COLORS["two"])


def metric_summary(df: pd.DataFrame, metrics: Iterable[str]) -> pd.DataFrame:
    rows = []
    for metric in metrics:
        grouped = df.groupby("combo", sort=False)[metric]
        for combo, values in grouped:
            values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
            n = len(values)
            mean = float(np.mean(values)) if n else np.nan
            std = float(np.std(values, ddof=1)) if n > 1 else 0.0
            sem = std / np.sqrt(n) if n else np.nan
            tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
            rows.append({
                "model": MODEL_NAME,
                "combo": combo,
                "metric": metric,
                "mean": mean,
                "std": std,
                "n": int(n),
                "sem": sem,
                "ci95_low": mean - tcrit * sem if n else np.nan,
                "ci95_high": mean + tcrit * sem if n else np.nan,
            })
    return pd.DataFrame(rows)


def paired_delta_vs_baseline(df: pd.DataFrame, metrics: dict[str, dict], baseline_combo: str = BASELINE_COMBO) -> pd.DataFrame:
    rows = []
    for metric, meta in metrics.items():
        if metric not in df.columns:
            continue
        pivot = df.pivot_table(index="repeat", columns="combo", values=metric, aggfunc="mean")
        if baseline_combo not in pivot.columns:
            raise KeyError(f"Baseline combo {baseline_combo!r} is absent for {metric}.")
        for combo in pivot.columns:
            if combo == baseline_combo:
                continue
            pair = pivot[[baseline_combo, combo]].dropna()
            if pair.empty:
                continue
            raw_delta = pair[combo] - pair[baseline_combo]
            signed_delta = raw_delta if meta["higher_is_better"] else -raw_delta
            n = len(signed_delta)
            mean = float(signed_delta.mean())
            std = float(signed_delta.std(ddof=1)) if n > 1 else 0.0
            sem = std / np.sqrt(n) if n else np.nan
            tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
            rows.append({
                "model": MODEL_NAME,
                "combo": combo,
                "metric": metric,
                "metric_label": meta["label"],
                "n_pairs": int(n),
                "mean_delta_vs_c": mean,
                "ci95_low": mean - tcrit * sem,
                "ci95_high": mean + tcrit * sem,
            })
    return pd.DataFrame(rows)


def save_current_figure(name: str) -> None:
    png = FIGURE_DIR / f"{name}.png"
    svg = FIGURE_DIR / f"{name}.svg"
    plt.savefig(png, bbox_inches="tight")
    plt.savefig(svg, bbox_inches="tight")
    print("Saved:", png)

## Load Completed Runs

In [ ]:
if USE_PRECOMPUTED_RESULTS and RAW_RESULTS_PATH.exists():
    raw_all = pd.read_parquet(RAW_RESULTS_PATH)
    print("Loaded precomputed raw results:", RAW_RESULTS_PATH)
elif RUN_FULL_SWEEP:
    raise NotImplementedError(
        "Full model recomputation is intentionally not started by this analysis notebook. "
        "Restore the packaged parquet for the hand-off workflow, or port the original sweep code into this notebook before enabling recomputation."
    )
else:
    raise FileNotFoundError(
        f"Missing precomputed raw results: {RAW_RESULTS_PATH}. "
        "Place feature_set_expansion_raw_results.parquet in precomputed_results/ or set RUN_FULL_SWEEP=True after adding sweep code."
    )

required_columns = {"model", "combo", "repeat", *METRICS.keys()}
missing = required_columns - set(raw_all.columns)
if missing:
    raise KeyError(f"Raw results are missing required columns: {sorted(missing)}")

runs = raw_all.loc[raw_all["model"].astype(str).str.lower() == MODEL_NAME.lower()].copy()
if runs.empty:
    raise ValueError(f"No rows found for model={MODEL_NAME!r} in {RAW_RESULTS_PATH}.")

runs["combo"] = runs["combo"].astype(str)
runs["repeat"] = pd.to_numeric(runs["repeat"], errors="raise").astype(int)
runs = runs.sort_values(["combo", "repeat"]).reset_index(drop=True)

coverage = runs.groupby("combo").agg(
    n_runs=("repeat", "count"),
    min_repeat=("repeat", "min"),
    max_repeat=("repeat", "max"),
).reset_index()
if "n_features" in runs.columns:
    n_features = runs.groupby("combo", as_index=False)["n_features"].median()
    coverage = coverage.merge(n_features, on="combo", how="left")
coverage.to_csv(TABLE_DIR / f"{MODEL_SLUG}_coverage.csv", index=False)

print("Rows:", runs.shape)
print("Feature-set combinations:", runs["combo"].nunique())
display(coverage.sort_values(["n_runs", "combo"], ascending=[False, True]))

## Summary Tables

In [ ]:
all_metric_names = list(METRICS) + [metric for metric in THRESHOLD_METRICS if metric in runs.columns]
summary = metric_summary(runs, all_metric_names)
summary.to_csv(TABLE_DIR / f"{MODEL_SLUG}_summary_by_combo_metric.csv", index=False)

probability_deltas = paired_delta_vs_baseline(runs, METRICS, baseline_combo=BASELINE_COMBO)
probability_deltas.to_csv(TABLE_DIR / f"{MODEL_SLUG}_paired_delta_vs_c.csv", index=False)

threshold_deltas = paired_delta_vs_baseline(runs, THRESHOLD_METRICS, baseline_combo=BASELINE_COMBO)
threshold_deltas.to_csv(TABLE_DIR / f"{MODEL_SLUG}_threshold_paired_delta_vs_c.csv", index=False)

wide_summary = (
    summary[summary["metric"].isin(METRICS)]
    .pivot(index="combo", columns="metric", values="mean")
    .reset_index()
)
wide_summary = wide_summary.sort_values(["test_PR_AUC", "test_LogLoss"], ascending=[False, True]).reset_index(drop=True)
wide_summary.to_csv(TABLE_DIR / f"{MODEL_SLUG}_probability_metric_leaderboard.csv", index=False)

threshold_summary = summary[summary["metric"].isin(THRESHOLD_METRICS)].copy()
if not threshold_summary.empty:
    threshold_wide_summary = (
        threshold_summary
        .pivot(index="combo", columns="metric", values="mean")
        .reset_index()
    )
    threshold_rank_columns = [metric for metric in THRESHOLD_METRICS if metric in threshold_wide_summary.columns]
    threshold_wide_summary["mean_threshold_rank"] = threshold_wide_summary[threshold_rank_columns].rank(ascending=False, method="min").mean(axis=1)
    threshold_wide_summary = threshold_wide_summary.sort_values("mean_threshold_rank").reset_index(drop=True)
else:
    threshold_wide_summary = pd.DataFrame()
threshold_wide_summary.to_csv(TABLE_DIR / f"{MODEL_SLUG}_threshold_metric_leaderboard.csv", index=False)

display(wide_summary.head(15))
if not threshold_wide_summary.empty:
    display(threshold_wide_summary.head(15))
display(probability_deltas.sort_values(["metric", "mean_delta_vs_c"], ascending=[True, False]).head(20))
if not threshold_deltas.empty:
    display(threshold_deltas.sort_values(["metric", "mean_delta_vs_c"], ascending=[True, False]).head(20))

## Standardized Figures

In [ ]:
def plot_paired_delta_grid(delta_table: pd.DataFrame, metrics: dict[str, dict], *, title: str, filename: str, ncols: int = 2) -> None:
    available_metrics = [metric for metric in metrics if metric in set(delta_table["metric"])]
    if not available_metrics:
        print(f"No metrics available for {filename}.")
        return

    nrows = int(np.ceil(len(available_metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.9 * ncols, 4.25 * nrows), sharex=False)
    axes = np.atleast_1d(axes).ravel()

    for ax, metric in zip(axes, available_metrics):
        sub = delta_table[delta_table["metric"] == metric].sort_values("mean_delta_vs_c")
        y = np.arange(len(sub))
        colors = np.where(
            sub["ci95_low"].to_numpy(float) > 0,
            COLORS["accent"],
            np.where(sub["ci95_high"].to_numpy(float) < 0, COLORS["negative"], "#6B7280"),
        )
        ax.hlines(y, sub["ci95_low"], sub["ci95_high"], color=COLORS["ci"], lw=2.0)
        ax.scatter(sub["mean_delta_vs_c"], y, color=colors, edgecolor="white", linewidth=0.7, zorder=3)
        ax.axvline(0, color="#374151", lw=1.0)
        ax.set_yticks(y)
        ax.set_yticklabels([combo_label(combo) for combo in sub["combo"]])
        ax.set_title(metrics[metric]["label"])
        ax.set_xlabel("Advantage vs c110 baseline")

    for ax in axes[len(available_metrics):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    save_current_figure(filename)
    plt.show()


probability_metric_order = ["test_PR_AUC", "test_ROC_AUC", "test_LogLoss", "test_Brier"]
metric_labels = [METRICS[metric]["label"] for metric in probability_metric_order]

plot_paired_delta_grid(
    probability_deltas,
    METRICS,
    title=f"{MODEL_NAME}: probability-metric advantage over c110",
    filename=f"{MODEL_SLUG}_paired_delta_vs_c",
    ncols=2,
)

plot_paired_delta_grid(
    threshold_deltas,
    THRESHOLD_METRICS,
    title=f"{MODEL_NAME}: threshold-metric advantage over c110",
    filename=f"{MODEL_SLUG}_threshold_paired_delta_vs_c",
    ncols=3,
)

# Rank overview for the probability metrics.
ranked = wide_summary.set_index("combo")[probability_metric_order].copy()
ranked["test_PR_AUC"] = ranked["test_PR_AUC"].rank(ascending=False, method="min")
ranked["test_ROC_AUC"] = ranked["test_ROC_AUC"].rank(ascending=False, method="min")
ranked["test_LogLoss"] = ranked["test_LogLoss"].rank(ascending=True, method="min")
ranked["test_Brier"] = ranked["test_Brier"].rank(ascending=True, method="min")
ranked["mean_rank"] = ranked[probability_metric_order].mean(axis=1)
ranked = ranked.sort_values("mean_rank")
rank_matrix = ranked[probability_metric_order].to_numpy(float)

fig, ax = plt.subplots(figsize=(7.8, 6.6))
im = ax.imshow(rank_matrix, cmap="viridis_r", aspect="auto", vmin=1, vmax=max(2, len(ranked)))
ax.set_xticks(np.arange(len(probability_metric_order)))
ax.set_xticklabels(metric_labels, rotation=20, ha="right")
ax.set_yticks(np.arange(len(ranked)))
ax.set_yticklabels([combo_label(combo) for combo in ranked.index])
ax.set_title(f"{MODEL_NAME}: rank by probability metric")
for row in range(rank_matrix.shape[0]):
    for col in range(rank_matrix.shape[1]):
        ax.text(col, row, f"{int(rank_matrix[row, col])}", ha="center", va="center", color="white", fontsize=8.5)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Rank; 1 is best")
fig.tight_layout()
save_current_figure(f"{MODEL_SLUG}_metric_rank_heatmap")
plt.show()

# Rank overview for the threshold metrics.
if not threshold_wide_summary.empty:
    threshold_metric_order = [metric for metric in THRESHOLD_METRICS if metric in threshold_wide_summary.columns]
    threshold_labels = [THRESHOLD_METRICS[metric]["label"] for metric in threshold_metric_order]

    threshold_ranked = threshold_wide_summary.set_index("combo")[threshold_metric_order].copy()
    for metric in threshold_metric_order:
        threshold_ranked[metric] = threshold_ranked[metric].rank(ascending=False, method="min")
    threshold_ranked["mean_rank"] = threshold_ranked[threshold_metric_order].mean(axis=1)
    threshold_ranked = threshold_ranked.sort_values("mean_rank")
    threshold_rank_matrix = threshold_ranked[threshold_metric_order].to_numpy(float)

    fig, ax = plt.subplots(figsize=(10.4, 6.8))
    im = ax.imshow(threshold_rank_matrix, cmap="viridis_r", aspect="auto", vmin=1, vmax=max(2, len(threshold_ranked)))
    ax.set_xticks(np.arange(len(threshold_metric_order)))
    ax.set_xticklabels(threshold_labels, rotation=25, ha="right")
    ax.set_yticks(np.arange(len(threshold_ranked)))
    ax.set_yticklabels([combo_label(combo) for combo in threshold_ranked.index])
    ax.set_title(f"{MODEL_NAME}: rank by threshold metric")
    for row in range(threshold_rank_matrix.shape[0]):
        for col in range(threshold_rank_matrix.shape[1]):
            ax.text(col, row, f"{int(threshold_rank_matrix[row, col])}", ha="center", va="center", color="white", fontsize=8.0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Rank; 1 is best")
    fig.tight_layout()
    save_current_figure(f"{MODEL_SLUG}_threshold_metric_rank_heatmap")
    plt.show()
